# 01. Exploratory Data Analysis & Business Insights

**Project:** ChurnGuard (Telco Customer Churn Prediction & Retention Optimization)  
**Author / Lead:** Senior ML Engineer & PM  
**Dataset:** IBM Telco Customer Churn (`NusaTel` Framing) & data.gov.my Cellular Trends  

---

## 1. Executive Summary & Objectives
This notebook conducts systematic Exploratory Data Analysis (EDA) per `docs/05_DATA_SPEC.md` section 6.

### Core Questions Addressed:
1. What is the baseline churn rate and target distribution?
2. Which demographic, account, and service factors have the strongest association with churn?
3. How do tenure, monthly charges, and lifetime value interact with customer retention?
4. What are the key business insights to drive proactive retention targeting?
5. How does this align with real-world Malaysian telecommunications trends (data.gov.my)?

In [1]:
# Setup and imports
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from churnguard.config import CFG, SEED
from churnguard.data.load import load_malaysia_subscribers

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print(f"Config loaded. Random seed: {SEED}")

## 2. Load Processed Training Split
> **Rule (CLAUDE.md):** No data leakage. We perform all exploratory data analysis strictly on the **training set** (`data/processed/train.parquet`, N = 4,930, 70% split).

In [2]:
train_path = CFG["paths"]["processed_dir"] / "train.parquet"
df_train = pd.read_parquet(train_path)
print(f"Training dataset shape: {df_train.shape[0]:,} rows x {df_train.shape[1]} columns")
df_train.head(5)

## 3. Target Distribution & Class Imbalance
Let's inspect the baseline churn rate across the training cohort.

In [3]:
churn_counts = df_train["Churn"].value_counts()
churn_pcts = df_train["Churn"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({"Count": churn_counts, "Percentage (%)": churn_pcts.round(2)})
target_summary.index = ["Retained (0)", "Churned (1)"]
display(target_summary)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Retained (0)", "Churned (1)"],
    churn_counts,
    color=["#2ecc71", "#e74c3c"],
    width=0.5,
    edgecolor="black",
)
for bar, pct in zip(bars, churn_pcts, strict=False):
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 50,
        f"{int(yval):,} ({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax.set_title("Target Distribution (Train: N = 4,930)", fontsize=13, fontweight="bold")
ax.set_ylabel("Customer Count")
ax.set_ylim(0, max(churn_counts) * 1.15)
plt.tight_layout()
plt.show()

## 4. Categorical Feature Analysis & Churn Rates
We analyze the churn rate across contract types, internet services, add-on protections, payment methods, and billing options.

In [4]:
cat_cols = [
    "Contract",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
    "PaymentMethod",
    "PaperlessBilling",
    "SeniorCitizen",
    "Partner",
    "Dependents",
]

for col in cat_cols:
    table = (
        df_train.groupby(col)["Churn"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "churn_rate"})
    )
    table["churn_rate_%"] = (table["churn_rate"] * 100).round(2)
    table["share_%"] = (table["count"] / len(df_train) * 100).round(2)
    print(f"=== Churn Rates by {col} ===")
    display(table)
    print("\n")

## 5. Numeric Feature Distributions by Churn Status
Let's evaluate `tenure`, `MonthlyCharges`, and `TotalCharges` across churners vs retained customers.

In [5]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
for col in num_cols:
    print(f"=== Summary Statistics: {col} ===")
    display(df_train.groupby("Churn")[col].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, col in enumerate(num_cols):
    sns.boxplot(
        data=df_train,
        x="Churn",
        y=col,
        hue="Churn",
        legend=False,
        ax=axes[idx],
        palette=["#2ecc71", "#e74c3c"],
        showmeans=True,
    )
    axes[idx].set_xticks([0, 1])
    axes[idx].set_xticklabels(["Retained (0)", "Churned (1)"])
    axes[idx].set_title(f"{col} by Churn Status", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Interaction Analysis: Tenure Buckets × Contract Type
Contract flexibility combined with early customer lifecycle creates critical risk clusters.

In [6]:
bins = [-1, 6, 12, 24, 48, 72]
labels = ["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]
df_analysis = df_train.copy()
df_analysis["tenure_bucket"] = pd.cut(df_analysis["tenure"], bins=bins, labels=labels)

pivot_rates = (
    df_analysis.pivot_table(
        index="Contract", columns="tenure_bucket", values="Churn", aggfunc="mean", observed=False
    )
    * 100
)
pivot_counts = df_analysis.pivot_table(
    index="Contract", columns="tenure_bucket", values="Churn", aggfunc="count", observed=False
)

display(Markdown("### Churn Rate (%) by Contract & Tenure Bucket"))
display(pivot_rates.round(2))
display(Markdown("### Customer Count (n) by Contract & Tenure Bucket"))
display(pivot_counts)

## 7. Malaysia Telecommunications Macro Context
Using data from `data.gov.my` (Malaysian Cellular Subscribers by Plan Type), we evaluate national postpaid vs prepaid trends.

In [7]:
my_df = load_malaysia_subscribers()
my_df["date"] = pd.to_datetime(my_df["date"])
my_df["year"] = my_df["date"].dt.year
my_pivot = my_df.pivot(index="year", columns="plan", values="subscriptions") / 1_000_000

display(my_pivot.tail(10).round(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    my_pivot.index,
    my_pivot["postpaid"],
    marker="o",
    linewidth=2.5,
    label="Postpaid (High ARPU)",
    color="#2980b9",
)
ax.plot(
    my_pivot.index,
    my_pivot["prepaid"],
    marker="s",
    linewidth=2.5,
    label="Prepaid (High Volume)",
    color="#e67e22",
)
ax.plot(
    my_pivot.index,
    my_pivot["total"],
    marker="^",
    linewidth=2.0,
    linestyle="--",
    label="Total Subscriptions",
    color="#2c3e50",
)
ax.set_title("Malaysia Cellular Subscriptions Trend (data.gov.my)", fontsize=13, fontweight="bold")
ax.set_ylabel("Subscriptions (Millions)")
ax.set_xlabel("Year")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Strategic Business Insights

Based on our statistical analysis, we establish **6 core business insights**:

1. **Contract Lock-in is the Single Strongest Churn Predictor:**
   - Month-to-month contracts have a **42.9% churn rate**, compared to **10.9% for 1-year** and **3.0% for 2-year** contracts.
   - *Action:* Offer tenure upgrade promotions and discounted 12-month contract migrations during early life stages.

2. **Critical First 90 Days / 6 Months Risk Window:**
   - Customers in months 0-6 with month-to-month contracts have a **53.1% churn rate**.
   - Median tenure for churners is only **10.0 months** vs **38.0 months** for retained users.
   - *Action:* Deploy dedicated proactive onboarding communications and check-ins at days 14, 30, and 60.

3. **Fiber Optic Service Onboarding & Support Deficit:**
   - Fiber optic users experience **41.8% churn** (vs **19.2% for DSL**), especially when lacking Tech Support (>48% churn).
   - *Action:* Bundle free Tech Support / VIP care for all new Fiber Optic activations.

4. **Payment Friction & Electronic Check Vulnerability:**
   - Customers paying by Electronic Check show a **45.6% churn rate**, while auto-pay methods have churn rates under **15-16%**.
   - *Action:* Incentivize credit card / bank direct debit auto-pay with a monthly bill rebate (e.g. RM 5/month discount).

5. **Protection Add-ons Dramatically Increase Stickiness:**
   - Customers lacking Online Security (42.0% churn) or Tech Support (41.9% churn) churn at 3x the rate of protected users (14.7% - 15.4%).
   - *Action:* Package standard security and backup add-ons into core high-speed tiers.

6. **Postpaid Protection Maximizes Long-Term Customer Lifetime Value (CLV):**
   - National Malaysian data shows steady growth in postpaid plans (>10M subscribers) where contractual retention directly preserves consistent ARPU.
   - *Action:* Target retention intervention on high-ARPU postpaid subscribers with high predicted churn probability.